# CineSense Agent — build real vector-search + PyTorch models

Run this in Google Colab (Runtime > Run all) if `build_embeddings.py` / `train_rating_model.py`
fail to run locally (torch/pandas blocked by a local Windows Application Control policy).

**Steps:** upload the two TMDB CSVs from this repo's `data/` folder when prompted, run every
cell, then download `cinesense_models.zip` and extract it into your local `models/` folder.

In [ ]:
!pip install -q faiss-cpu sentence-transformers
import os
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [ ]:
from google.colab import files
print('Upload tmdb_5000_movies.csv and tmdb_5000_credits.csv (from this repo\'s data/ folder)')
uploaded = files.upload()
import shutil
for name in uploaded:
    shutil.move(name, f'data/{name}')

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'all-MiniLM-L6-v2'

def names(text, limit=None):
    items = [x['name'] for x in json.loads(text)]
    return items[:limit] if limit else items

def director(text):
    for person in json.loads(text):
        if person['job'] == 'Director':
            return person['name']
    return ''

movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')
df = movies.merge(credits[['movie_id', 'cast', 'crew']], left_on='id', right_on='movie_id')

df['genre_list'] = df['genres'].apply(names)
df['keyword_list'] = df['keywords'].apply(names)
df['top_cast'] = df['cast'].apply(lambda t: names(t, 3))
df['director'] = df['crew'].apply(director)
df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['overview'] = df['overview'].fillna('')

def make_tags(row):
    parts = [row['overview']]
    if row['genre_list']:
        parts.append('Genres: ' + ', '.join(row['genre_list']) + '.')
    if row['keyword_list']:
        parts.append('Themes: ' + ', '.join(row['keyword_list'][:10]) + '.')
    if row['top_cast']:
        parts.append('Starring ' + ', '.join(row['top_cast']) + '.')
    if row['director']:
        parts.append('Directed by ' + row['director'] + '.')
    return ' '.join(p for p in parts if p)

df['tags'] = df.apply(make_tags, axis=1)
catalog = df[['id', 'title', 'year', 'genre_list', 'vote_average', 'vote_count',
              'overview', 'director', 'top_cast', 'tags']].reset_index(drop=True)
print(f'Loaded {len(catalog)} movies')

In [ ]:
model = SentenceTransformer(EMBED_MODEL)
embeddings = model.encode(
    catalog['tags'].tolist(), batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

joblib.dump(catalog.drop(columns=['tags']), 'models/catalog.pkl')
np.save('models/embeddings.npy', embeddings)
faiss.write_index(index, 'models/movie_index.faiss')
print(f'Vector index ready: {index.ntotal} movies, dim={embeddings.shape[1]} ({EMBED_MODEL})')

## PyTorch rating model (vs. Random Forest baseline)

In [ ]:
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURES = ['budget_millions', 'runtime', 'year', 'main_genre']

train = df[(df['budget'] > 0) & (df['runtime'] > 0) & (df['vote_count'] >= 50)].dropna(subset=['year']).copy()
train['main_genre'] = train['genre_list'].apply(lambda g: g[0] if g else 'Unknown')
train['budget_millions'] = train['budget'] / 1_000_000
X, y = train[FEATURES], train['vote_average']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), ['budget_millions', 'runtime', 'year']),
    ('genre', OneHotEncoder(handle_unknown='ignore'), ['main_genre']),
])
X_train_enc = preprocess.fit_transform(X_train).astype('float32')
X_test_enc = preprocess.transform(X_test).astype('float32')
if hasattr(X_train_enc, 'toarray'):
    X_train_enc, X_test_enc = X_train_enc.toarray(), X_test_enc.toarray()

class RatingMLP(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

X_train_t = torch.from_numpy(X_train_enc)
y_train_t = torch.from_numpy(y_train.to_numpy(dtype='float32'))
X_test_t = torch.from_numpy(X_test_enc)

net = RatingMLP(n_inputs=X_train_enc.shape[1])
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

net.train()
for epoch in range(1, 301):
    optimizer.zero_grad()
    loss = loss_fn(net(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f'epoch {epoch:4d}  train MSE {loss.item():.4f}')

net.eval()
with torch.no_grad():
    torch_pred = net(X_test_t).numpy()
print(f"PyTorch MLP   R2: {r2_score(y_test, torch_pred):.3f}   MAE: {mean_absolute_error(y_test, torch_pred):.3f}")

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_enc, y_train)
rf_pred = rf.predict(X_test_enc)
print(f"Random Forest R2: {r2_score(y_test, rf_pred):.3f}   MAE: {mean_absolute_error(y_test, rf_pred):.3f}")

torch.save(net.state_dict(), 'models/rating_model_torch.pt')
joblib.dump({'preprocess': preprocess, 'n_inputs': X_train_enc.shape[1]},
            'models/rating_feature_pipeline.pkl')
print('Saved models/rating_model_torch.pt, models/rating_feature_pipeline.pkl')

In [ ]:
import shutil
shutil.make_archive('cinesense_models', 'zip', 'models')
from google.colab import files
files.download('cinesense_models.zip')
print('Extract this zip into your local cinesense-agent models/ folder.')